In [2]:
print("Howdy")

Howdy


In [9]:
import lancedb
import geneva
import pyarrow as pa

In [10]:
clip_dim = 3
tower_dim = 2

cover_schema = pa.schema(
    [
        pa.field("cover_id", pa.int64(), nullable=False),
        pa.field("book_id", pa.int64(), nullable=False),
        pa.field("isbn_13", pa.string(), nullable=False),
        pa.field("cover_url", pa.string(), nullable=False),
        pa.field("cover_embedding", pa.fixed_shape_tensor(pa.float32(), (clip_dim,)), nullable=False),
        pa.field("tower_embedding", pa.list_(pa.float32(), tower_dim), nullable=True),
    ]
)

In [30]:
uri = "test_lancedb"
db = lancedb.connect(uri)

In [6]:
cover_table = db.create_table(
    "covers", schema=cover_schema, exist_ok=True
)

cover_table

TypeError: Can't instantiate abstract class Table without an implementation for abstract methods 'blob_columns', 'fetch_blob_files', 'fetch_blob_ranges', 'fetch_blobs', 'tokenize'

In [7]:
cover_table.head()

NameError: name 'cover_table' is not defined

In [ ]:
cover_schema

In [ ]:
raw_videos = db.create_table(
    "raw_videos",
    schema=schema,
    primary_key="video_id"
)

In [31]:
import duckdb
import lancedb

db = lancedb.connect(uri)

# Extract the underlying Arrow/Lance datasets
t1 = db.open_table("interactions").to_lance()
t2 = db.open_table("covers").to_lance()

In [34]:
t1.head(5)

pyarrow.Table
user_id: extension<arrow.uuid> not null
cover_id: int64 not null
type: string not null
score: int64 not null
timestamp: timestamp[us] not null
----
user_id: [[2223305FAE134CBA9C4484B6BE364F96],[2223305FAE134CBA9C4484B6BE364F96],[B2C72CFC72184362A06A6D4852224CC9]]
cover_id: [[5],[1],[1]]
type: [["rating"],["rating"],["rating"]]
score: [[4],[4],[4]]
timestamp: [[2026-08-13 01:02:46.654273],[2026-08-13 01:06:54.856542],[2026-08-13 01:07:11.525132]]

In [35]:
t2.head(5)

pyarrow.Table
cover_id: int64 not null
book_id: int64 not null
isbn_13: string not null
cover_url: string not null
cover_embedding: extension<arrow.fixed_shape_tensor[value_type=float, shape=[3]]> not null
tower_embedding: fixed_size_list<item: float>[2]
  child 0, item: float
----
cover_id: [[1,5]]
book_id: [[2,2]]
isbn_13: [["1234567891011","1234567891014"]]
cover_url: [["something.cool.com/bruh.jpg","something.cool.com/bruh2.jpg"]]
cover_embedding: [[[1,3,9],[15,2,2]]]
tower_embedding: [[null,[3,2]]]

In [36]:
# DuckDB automatically recognizes the Python variables 't1' and 't2'
query = """
    SELECT t1.cover_id, t1.user_id, t1.score, t2.cover_embedding, t2.tower_embedding 
    FROM t1 
    JOIN t2 ON t1.cover_id = t2.cover_id
"""

# Execute the join and return a Pandas DataFrame
joined_df = duckdb.sql(query).pl()
joined_df

cover_id,user_id,score,cover_embedding,tower_embedding
i64,str,i64,"array[f32, 3]","array[f32, 2]"
1,"""b2c72cfc-7218-4362-a06a-6d4852…",4,"[1.0, 3.0, 9.0]",null
5,"""2223305f-ae13-4cba-9c44-84b6be…",4,"[15.0, 2.0, 2.0]","[3.0, 2.0]"
1,"""2223305f-ae13-4cba-9c44-84b6be…",4,"[1.0, 3.0, 9.0]",null


In [79]:
import duckdb

con = duckdb.connect()
con.execute("INSTALL lance; LOAD lance;")
con.execute("ATTACH './test_lancedb' AS lance_ns (TYPE LANCE);")

# Join two tables by id
result = con.execute("""
    SELECT feedback.cover_id, covers.cover_embedding
    FROM lance_ns.main.interactions AS feedback
    JOIN lance_ns.main.covers AS covers
      ON feedback.cover_id = covers.cover_id
""").pl()

In [80]:
result

cover_id,cover_embedding
i64,"array[f32, 3]"
1,"[1.0, 3.0, 9.0]"
5,"[15.0, 2.0, 2.0]"
1,"[1.0, 3.0, 9.0]"


In [65]:
result.to_torch()

TypeError: cannot convert DataFrame to Tensor (mixed type columns result in `object` dtype)
Schema({'cover_id': Int64, 'cover_embedding': Array(Float32, shape=(3,))})

In [44]:
result.with_row_index()

index,user_id,cover_id,type,score,timestamp,cover_id_1,book_id,isbn_13,cover_url,cover_embedding,tower_embedding
u32,str,i64,str,i64,datetime[μs],i64,i64,str,str,"array[f32, 3]","array[f32, 2]"
0,"""b2c72cfc-7218-4362-a06a-6d4852…",1,"""rating""",4,2026-08-13 01:07:11.525132,1,2,"""1234567891011""","""something.cool.com/bruh.jpg""","[1.0, 3.0, 9.0]",null
1,"""2223305f-ae13-4cba-9c44-84b6be…",5,"""rating""",4,2026-08-13 01:02:46.654273,5,2,"""1234567891014""","""something.cool.com/bruh2.jpg""","[15.0, 2.0, 2.0]","[3.0, 2.0]"
2,"""2223305f-ae13-4cba-9c44-84b6be…",1,"""rating""",4,2026-08-13 01:06:54.856542,1,2,"""1234567891011""","""something.cool.com/bruh.jpg""","[1.0, 3.0, 9.0]",null


In [46]:
import polars as pl

result.with_row_index().filter(pl.col("index").is_in([2, 1]))

index,user_id,cover_id,type,score,timestamp,cover_id_1,book_id,isbn_13,cover_url,cover_embedding,tower_embedding
u32,str,i64,str,i64,datetime[μs],i64,i64,str,str,"array[f32, 3]","array[f32, 2]"
1,"""2223305f-ae13-4cba-9c44-84b6be…",5,"""rating""",4,2026-08-13 01:02:46.654273,5,2,"""1234567891014""","""something.cool.com/bruh2.jpg""","[15.0, 2.0, 2.0]","[3.0, 2.0]"
2,"""2223305f-ae13-4cba-9c44-84b6be…",1,"""rating""",4,2026-08-13 01:06:54.856542,1,2,"""1234567891011""","""something.cool.com/bruh.jpg""","[1.0, 3.0, 9.0]",null


In [78]:
result["cover_id"].to_torch().unsqueeze(dim=-1)

tensor([[1],
        [5],
        [1]])

In [82]:
result["cover_embedding"].to_torch()

tensor([[ 1.,  3.,  9.],
        [15.,  2.,  2.],
        [ 1.,  3.,  9.]])

In [68]:
res = pl.DataFrame({"index": [2, 1]}).join(result.with_row_index(), on="index", how="left")
res

index,cover_id,cover_embedding
i64,i64,"array[f32, 3]"
2,1,"[1.0, 3.0, 9.0]"
1,5,"[15.0, 2.0, 2.0]"


In [69]:
res["cover_embedding"].to_torch()

tensor([[ 1.,  3.,  9.],
        [15.,  2.,  2.]])

In [71]:
res["cover_embedding"].to_torch().shape

torch.Size([2, 3])

In [61]:
import torch

torch.tensor([2]).repeat(3, 1)

tensor([[2],
        [2],
        [2]])